# ПОЛНЫЙ АНАЛИЗ КОЭФФИЦИЕНТОВ ПРОЛОНГАЦИИ

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
import warnings
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows
warnings.filterwarnings('ignore')

# настройки
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', None)
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10

# ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ

In [ ]:
# загрузка данных
financial_df = pd.read_csv('financial_data.csv', encoding='utf-8')
prolongations_df = pd.read_csv('prolongations.csv', encoding='utf-8')

print(f"financial_data.csv: {len(financial_df)} строк")
print(f"prolongations.csv: {len(prolongations_df)} строк")

# колонки месяцев
month_cols = [c for c in financial_df.columns 
              if c not in ['id', 'Причина дубля', 'Account'] 
              and any(ch.isdigit() for ch in str(c))]

print(f"Месяцы в финансовых данных: {len(month_cols)}")

# словарь для парсинга русских месяцев
MONTH_RU = {'январь': 1, 'февраль': 2, 'март': 3, 'апрель': 4, 'май': 5, 'июнь': 6, 'июль': 7, 'август': 8, 'сентябрь': 9, 'октябрь': 10, 'ноябрь': 11, 'декабрь': 12}

def parse_month_ru(s):
    """конвертирует 'Ноябрь 2022' → Timestamp(2022-11-01)"""
    if pd.isna(s):
        return None
    try:
        parts = str(s).lower().strip().split()
        if len(parts) >= 2:
            month_num = MONTH_RU.get(parts[0])
            year = int(parts[1])
            if month_num:
                return pd.Timestamp(year=year, month=month_num, day=1)
    except:
        pass
    return None

def clean_value(val):
    """очищает финансовое значение"""
    if pd.isna(val) or val == '':
        return 0.0, False
    
    s = str(val).strip().lower()
    
    if s in ['стоп', 'end']:
        return 0.0, True
    if s == 'в ноль':
        return 0.0, False
    
    try:
        cleaned = s.replace(' ', '').replace('\xa0', '').replace(',', '.')
        return float(cleaned), False
    except:
        return 0.0, False

# ПРЕДОБРАБОТКА ФИНАНСОВЫХ ДАННЫХ

In [ ]:
# применяем очистку к финансовым колонкам
for col in month_cols:
    financial_df[col + '_num'] = financial_df[col].apply(lambda x: clean_value(x)[0])
    financial_df[col + '_stop'] = financial_df[col].apply(lambda x: clean_value(x)[1])

# агрегация дублей по id
agg_rules = {col + '_num': 'sum' for col in month_cols}
agg_rules.update({col + '_stop': 'max' for col in month_cols})
agg_rules['Account'] = 'first'

financial_agg = financial_df.groupby('id', as_index=False).agg(agg_rules)

print(f"\nПосле агрегации: {len(financial_agg)} уникальных проектов")

# добавляем информацию из prolongations
prolongations_df['last_month_dt'] = prolongations_df['month'].apply(parse_month_ru)
id_to_info = prolongations_df.groupby('id').first()[['last_month_dt', 'AM']].to_dict('index')

financial_agg['last_month'] = financial_agg['id'].map(lambda x: id_to_info.get(x, {}).get('last_month_dt'))
financial_agg['AM'] = financial_agg['id'].map(lambda x: id_to_info.get(x, {}).get('AM'))

# если AM не найден, берём из финансовых данных
mask = financial_agg['AM'].isna()
financial_agg.loc[mask, 'AM'] = financial_agg.loc[mask, 'Account']
financial_agg['AM'] = financial_agg['AM'].fillna('Не назначен')


# исключаем только если 'стоп' в месяце <= last_month
def is_excluded_by_stop(row, last_month, month_cols):
    """проверяет, есть ли 'стоп' в месяце реализации или ранее"""
    if pd.isna(last_month):
        return False
    for col in month_cols:
        col_dt = parse_month_ru(col)
        if col_dt and col_dt <= last_month and row.get(col + '_stop', False):
            return True
    return False

# применяем проверку к каждому проекту
financial_agg['has_stop'] = financial_agg.apply(
    lambda row: is_excluded_by_stop(row, row['last_month'], month_cols),
    axis=1
)

# исключаем проекты со стоп
excluded = financial_agg['has_stop'].sum()
fin_clean = financial_agg[~financial_agg['has_stop']].copy()

print(f"Исключено проектов со 'стоп'/'end': {excluded}")

print(f"Исключено проектов со 'стоп'/'end': {excluded}")
print(f"Активных проектов: {len(fin_clean)}")
print(f"Менеджеров: {fin_clean['AM'].nunique()}")

# ФУНКЦИИ РАСЧЁТА КОЭФФИЦИЕНТОВ

In [ ]:
def get_col_for_month(dt):
    """находит название колонки по datetime"""
    if dt is None:
        return None
    for col in month_cols:
        col_dt = parse_month_ru(col)
        if col_dt and col_dt.normalize() == dt.normalize():
            return col + '_num'
    return None

def calculate_k1(df, report_month, manager=None):
    """расчёт K1 для отчётного месяца"""
    prev_month = report_month - pd.DateOffset(months=1)
    
    col_m = get_col_for_month(report_month)
    col_prev = get_col_for_month(prev_month)
    
    if not col_m or not col_prev:
        return None
    
    # фильтр по менеджеру
    if manager:
        df = df[df['AM'] == manager].copy()
    
    # проекты с last_month = prev_month
    mask = df['last_month'].apply(
        lambda x: pd.notna(x) and x.normalize() == prev_month.normalize()
    )
    candidates = df[mask].copy()
    
    if len(candidates) == 0:
        return None
    
    denom = candidates[col_prev].sum()
    prolonged = candidates[candidates[col_m] > 0]
    num = prolonged[col_m].sum() if len(prolonged) > 0 else 0
    
    if denom == 0:
        return None
    
    return {
        'month': report_month,
        'k1': num / denom,
        'denominator': denom,
        'numerator': num,
        'projects': len(candidates),
        'prolonged': len(prolonged)
    }

def calculate_k2(df, report_month, manager=None):
    """расчёт K2 для отчётного месяца"""
    prev_1 = report_month - pd.DateOffset(months=1)
    prev_2 = report_month - pd.DateOffset(months=2)
    
    col_m = get_col_for_month(report_month)
    col_p1 = get_col_for_month(prev_1)
    col_p2 = get_col_for_month(prev_2)
    
    if not col_m or not col_p1 or not col_p2:
        return None
    
    # фильтр по менеджеру
    if manager:
        df = df[df['AM'] == manager].copy()
    
    # проекты с last_month = M-2
    mask = df['last_month'].apply(
        lambda x: pd.notna(x) and x.normalize() == prev_2.normalize()
    )
    candidates = df[mask].copy()
    
    if len(candidates) == 0:
        return None
    
    # без отгрузки в M-1
    candidates = candidates[candidates[col_p1] == 0].copy()
    
    if len(candidates) == 0:
        return None
    
    denom = candidates[col_p2].sum()
    prolonged = candidates[candidates[col_m] > 0]
    num = prolonged[col_m].sum() if len(prolonged) > 0 else 0
    
    if denom == 0:
        return None
    
    return {
        'month': report_month,
        'k2': num / denom,
        'denominator': denom,
        'numerator': num,
        'projects': len(candidates),
        'prolonged': len(prolonged)
    }

# РАСЧЁТ ПО МЕСЯЦАМ И МЕНЕДЖЕРАМ

In [ ]:
report_months = pd.date_range('2023-01-01', '2023-12-01', freq='MS')
managers = sorted(fin_clean['AM'].unique())

print(f"\nРасчёт коэффициентов...")
print(f"Месяцев: {len(report_months)}")
print(f"Менеджеров: {len(managers)}")

# результаты
k1_dept = []
k2_dept = []
k1_managers = []
k2_managers = []

# по отделу
for month in report_months:
    k1 = calculate_k1(fin_clean, month)
    if k1:
        k1_dept.append(k1)
    
    k2 = calculate_k2(fin_clean, month)
    if k2:
        k2_dept.append(k2)

# по менеджерам
for manager in managers:
    for month in report_months:
        k1 = calculate_k1(fin_clean, month, manager)
        if k1:
            k1_managers.append({**k1, 'manager': manager})
        
        k2 = calculate_k2(fin_clean, month, manager)
        if k2:
            k2_managers.append({**k2, 'manager': manager})

# конвертация в DataFrame
k1_dept_df = pd.DataFrame(k1_dept)
k2_dept_df = pd.DataFrame(k2_dept)
k1_mgr_df = pd.DataFrame(k1_managers)
k2_mgr_df = pd.DataFrame(k2_managers)

print(f"K1 по отделу: {len(k1_dept_df)} месяцев")
print(f"K2 по отделу: {len(k2_dept_df)} месяцев")
print(f"K1 по менеджерам: {len(k1_mgr_df)} записей")
print(f"K2 по менеджерам: {len(k2_mgr_df)} записей")

# ГОДОВЫЕ ИТОГИ 

In [ ]:
def calc_weighted_avg(df, coef_col):
    """средневзвешенный коэффициент"""
    if len(df) == 0 or df['denominator'].sum() == 0:
        return None
    return df['numerator'].sum() / df['denominator'].sum()

# по отделу
k1_year_dept = calc_weighted_avg(k1_dept_df, 'k1')
k2_year_dept = calc_weighted_avg(k2_dept_df, 'k2')

print(f"\nГодовые коэффициенты (отдел):")
print(f"   K1 (2023): {k1_year_dept:.1%}" if k1_year_dept else "   K1 (2023): N/A")
print(f"   K2 (2023): {k2_year_dept:.1%}" if k2_year_dept else "   K2 (2023): N/A")

# по менеджерам
manager_stats = []
for manager in managers:
    k1_mgr = k1_mgr_df[k1_mgr_df['manager'] == manager]
    k2_mgr = k2_mgr_df[k2_mgr_df['manager'] == manager]
    
    manager_stats.append({
        'manager': manager,
        'k1_year': calc_weighted_avg(k1_mgr, 'k1'),
        'k2_year': calc_weighted_avg(k2_mgr, 'k2'),
        'k1_months': len(k1_mgr),
        'k2_months': len(k2_mgr),
        'total_projects': k1_mgr['projects'].sum(),
        'total_prolonged': k1_mgr['prolonged'].sum(),
        'k1_denom': k1_mgr['denominator'].sum(),
        'k1_numer': k1_mgr['numerator'].sum(),
    })

manager_stats_df = pd.DataFrame(manager_stats)
manager_stats_df = manager_stats_df.sort_values('k1_year', ascending=False)

print(f"\nТоп-3 менеджера по K1:")
for idx, row in manager_stats_df.head(3).iterrows():
    print(f"{row['manager']}: {row['k1_year']:.1%}" if pd.notna(row['k1_year']) else f"{row['manager']}: N/A")

# ВИЗУАЛИЗАЦИЯ

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# график 1. Динамика K1 по отделу
ax1 = axes[0, 0]
if len(k1_dept_df) > 0:
    k1_sorted = k1_dept_df.sort_values('month')
    ax1.plot(k1_sorted['month'], k1_sorted['k1'], marker='o', linewidth=2.5, color='#2E86AB', markersize=8)
    if k1_year_dept:
        ax1.axhline(y=k1_year_dept, color='#E74C3C', linestyle='--', linewidth=2, label=f'Среднее: {k1_year_dept:.1%}')
    ax1.set_title('Динамика K1 по отделу (2023)', fontsize=14, fontweight='bold', pad=15)
    ax1.set_xlabel('Месяц', fontsize=11)
    ax1.set_ylabel('Коэффициент', fontsize=11)
    ax1.legend(loc='best')
    ax1.grid(True, alpha=0.3)
    ax1.set_ylim([0, 1])
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
    from matplotlib.dates import DateFormatter
    ax1.xaxis.set_major_formatter(DateFormatter('%Y-%m'))

# график 2. Динамика K2 по отделу
ax2 = axes[0, 1]
if len(k2_dept_df) > 0:
    k2_sorted = k2_dept_df.sort_values('month')
    ax2.plot(k2_sorted['month'], k2_sorted['k2'], marker='s', linewidth=2.5, color='#A23B72', markersize=8)
    if k2_year_dept:
        ax2.axhline(y=k2_year_dept, color='#E74C3C', linestyle='--', linewidth=2, label=f'Среднее: {k2_year_dept:.1%}')
    ax2.set_title('Динамика K2 по отделу (2023)', fontsize=14, fontweight='bold', pad=15)
    ax2.set_xlabel('Месяц', fontsize=11)
    ax2.set_ylabel('Коэффициент', fontsize=11)
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim([0, 1])
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
    ax2.xaxis.set_major_formatter(DateFormatter('%Y-%m'))

# график 3. Рейтинг менеджеров по K1
ax3 = axes[1, 0]
mgr_sorted = manager_stats_df.dropna(subset=['k1_year']).sort_values('k1_year')
if len(mgr_sorted) > 0:
    colors = ['#2ECC71' if x >= k1_year_dept else '#E74C3C' for x in mgr_sorted['k1_year']] if k1_year_dept else '#3498DB'
    bars = ax3.barh(mgr_sorted['manager'], mgr_sorted['k1_year'], color=colors, alpha=0.8)
    if k1_year_dept:
        ax3.axvline(x=k1_year_dept, color='red', linestyle='--', linewidth=2, label=f'Среднее: {k1_year_dept:.1%}')
    ax3.set_title('Рейтинг менеджеров по K1 (2023)', fontsize=14, fontweight='bold', pad=15)
    ax3.set_xlabel('Коэффициент K1', fontsize=11)
    ax3.legend(loc='best')
    ax3.grid(True, alpha=0.3, axis='x')

# график 4. Количество проектов по менеджерам
ax4 = axes[1, 1]
if len(manager_stats_df) > 0:
    ax4.barh(manager_stats_df['manager'], manager_stats_df['total_projects'], color='#3498DB', alpha=0.8)
    ax4.set_title('Количество проектов по менеджерам', fontsize=14, fontweight='bold', pad=15)
    ax4.set_xlabel('Количество проектов', fontsize=11)
    ax4.grid(True, alpha=0.3, axis='x')

plt.tight_layout(pad=3.0)
plt.savefig('prolongation_analysis_2023.png', dpi=300, bbox_inches='tight')
plt.show()

# ЭКСПОРТ В EXCEL

In [ ]:
wb = Workbook()

# вкладка 1: Сводка
ws_summary = wb.active
ws_summary.title = 'Сводка'

ws_summary['A1'] = 'ОТЧЁТ ПО КОЭФФИЦИЕНТАМ ПРОЛОНГАЦИИ ДОГОВОРОВ'
ws_summary['A1'].font = Font(bold=True, size=16, color='1F4E79')
ws_summary.merge_cells('A1:G1')

ws_summary['A2'] = f'Период: 2023 год | Дата формирования: {datetime.now().strftime("%d.%m.%Y")}'
ws_summary['A2'].font = Font(italic=True, size=11)

# основные показатели
ws_summary['A4'] = 'ОСНОВНЫЕ ПОКАЗАТЕЛИ ПО ОТДЕЛУ'
ws_summary['A4'].font = Font(bold=True, size=12, color='2E86AB')

summary_data = [
    ['Показатель', 'Значение'],
    ['K1 за год (средневзвешенный)', f'{k1_year_dept:.1%}' if k1_year_dept else 'N/A'],
    ['K2 за год (средневзвешенный)', f'{k2_year_dept:.1%}' if k2_year_dept else 'N/A'],
    ['Всего активных проектов', len(fin_clean)],
    ['Исключено (стоп/end)', int(excluded)],
    ['Всего менеджеров', fin_clean['AM'].nunique()],
    ['Всего пролонгировано (K1)', int(k1_dept_df['prolonged'].sum()) if len(k1_dept_df) > 0 else 0],
    ['Общий объём отгрузок (знаменатель K1)', f'{k1_dept_df["denominator"].sum():,.0f}' if len(k1_dept_df) > 0 else '0'],
]

for row_idx, row_data in enumerate(summary_data, start=6):
    for col_idx, value in enumerate(row_data, start=1):
        cell = ws_summary.cell(row=row_idx, column=col_idx, value=value)
        cell.font = Font(size=11)
        if row_idx == 6:
            cell.font = Font(bold=True, size=11)

# помесячные данные K1
ws_summary['A17'] = 'K1 ПО МЕСЯЦАМ'
ws_summary['A17'].font = Font(bold=True, size=12, color='2E86AB')

headers = ['Месяц', 'K1', 'Числитель', 'Знаменатель', 'Проектов', 'Пролонгировано', '% пролонгации']
for col_idx, header in enumerate(headers, start=1):
    cell = ws_summary.cell(row=18, column=col_idx, value=header)
    cell.font = Font(bold=True, size=10)
    cell.fill = PatternFill(start_color='D6EAF8', end_color='D6EAF8', fill_type='solid')

if len(k1_dept_df) > 0:
    for row_idx, row in k1_dept_df.sort_values('month').iterrows():
        ws_summary.cell(row=19+row_idx, column=1, value=row['month'].strftime('%Y-%m'))
        ws_summary.cell(row=19+row_idx, column=2, value=f"{row['k1']:.2%}")
        ws_summary.cell(row=19+row_idx, column=3, value=f"{row['numerator']:,.0f}")
        ws_summary.cell(row=19+row_idx, column=4, value=f"{row['denominator']:,.0f}")
        ws_summary.cell(row=19+row_idx, column=5, value=int(row['projects']))
        ws_summary.cell(row=19+row_idx, column=6, value=int(row['prolonged']))
        ws_summary.cell(row=19+row_idx, column=7, value=f"{row['prolonged']/row['projects']:.1%}" if row['projects'] > 0 else '0%')
        
        # цветовое выделение
        k1_cell = ws_summary.cell(row=19+row_idx, column=2)
        if k1_year_dept:
            if row['k1'] >= k1_year_dept:
                k1_cell.fill = PatternFill(start_color='D5F5E3', end_color='D5F5E3', fill_type='solid')
            else:
                k1_cell.fill = PatternFill(start_color='FADBD8', end_color='FADBD8', fill_type='solid')

# вкладка 2. По менеджерам 
ws_managers = wb.create_sheet('По менеджерам')

ws_managers['A1'] = 'РЕЙТИНГ МЕНЕДЖЕРОВ ПО ЭФФЕКТИВНОСТИ ПРОЛОНГАЦИИ'
ws_managers['A1'].font = Font(bold=True, size=14, color='1F4E79')
ws_managers.merge_cells('A1:I1')

headers = ['Менеджер', 'K1 (год)', 'K2 (год)', 'K1 месяцев', 'K2 месяцев', 
           'Всего проектов', 'Пролонгировано', '% пролонгации', 'Объём (K1)']
for col_idx, header in enumerate(headers, start=1):
    cell = ws_managers.cell(row=3, column=col_idx, value=header)
    cell.font = Font(bold=True, size=10)
    cell.fill = PatternFill(start_color='D6EAF8', end_color='D6EAF8', fill_type='solid')

for row_idx, row in manager_stats_df.iterrows():
    ws_managers.cell(row=4+row_idx, column=1, value=row['manager'])
    ws_managers.cell(row=4+row_idx, column=2, value=f"{row['k1_year']:.1%}" if pd.notna(row['k1_year']) else 'N/A')
    ws_managers.cell(row=4+row_idx, column=3, value=f"{row['k2_year']:.1%}" if pd.notna(row['k2_year']) else 'N/A')
    ws_managers.cell(row=4+row_idx, column=4, value=int(row['k1_months']))
    ws_managers.cell(row=4+row_idx, column=5, value=int(row['k2_months']))
    ws_managers.cell(row=4+row_idx, column=6, value=int(row['total_projects']))
    ws_managers.cell(row=4+row_idx, column=7, value=int(row['total_prolonged']))
    ws_managers.cell(row=4+row_idx, column=8, value=f"{row['total_prolonged']/row['total_projects']:.1%}" if row['total_projects'] > 0 else '0%')
    ws_managers.cell(row=4+row_idx, column=9, value=f"{row['k1_denom']:,.0f}")
    
    # цветовое выделение K1
    k1_cell = ws_managers.cell(row=4+row_idx, column=2)
    if pd.notna(row['k1_year']) and k1_year_dept:
        if row['k1_year'] >= k1_year_dept:
            k1_cell.fill = PatternFill(start_color='D5F5E3', end_color='D5F5E3', fill_type='solid')
        else:
            k1_cell.fill = PatternFill(start_color='FADBD8', end_color='FADBD8', fill_type='solid')

# вкладка 3. Динамика 
ws_dynamics = wb.create_sheet('Динамика')

ws_dynamics['A1'] = 'ПОМЕСЯЧНАЯ ДИНАМИКА K1 ПО МЕНЕДЖЕРАМ'
ws_dynamics['A1'].font = Font(bold=True, size=14, color='1F4E79')
ws_dynamics.merge_cells('A1:M1')

# Pivot table
k1_pivot = k1_mgr_df.pivot_table(index='manager', columns='month', values='k1', aggfunc='first')
k1_pivot = k1_pivot.reset_index()

# заголовки
ws_dynamics.cell(row=3, column=1, value='Менеджер')
for col_idx, month in enumerate(report_months, start=2):
    ws_dynamics.cell(row=3, column=col_idx, value=month.strftime('%Y-%m'))
    ws_dynamics.cell(row=3, column=col_idx).font = Font(bold=True, size=9)
    ws_dynamics.cell(row=3, column=col_idx).fill = PatternFill(start_color='D6EAF8', end_color='D6EAF8', fill_type='solid')

# данные
for row_idx, manager in enumerate(k1_pivot['manager'], start=4):
    ws_dynamics.cell(row=row_idx, column=1, value=manager)
    for col_idx, month in enumerate(report_months, start=2):
        val = k1_pivot[k1_pivot['manager'] == manager][month].values
        if len(val) > 0 and pd.notna(val[0]):
            ws_dynamics.cell(row=row_idx, column=col_idx, value=f"{val[0]:.2%}")
        else:
            ws_dynamics.cell(row=row_idx, column=col_idx, value='')

# Вкладка 4. Методология 
ws_method = wb.create_sheet('Методология')

ws_method['A1'] = 'МЕТОДОЛОГИЯ РАСЧЁТА КОЭФФИЦИЕНТОВ ПРОЛОНГАЦИИ'
ws_method['A1'].font = Font(bold=True, size=14, color='1F4E79')
ws_method.merge_cells('A1:E1')

methodology_text = [
    ['', ''],
    ['ФОРМУЛЫ РАСЧЁТА', ''],
    ['', ''],
    ['Коэффициент K1 (пролонгация в первый месяц):', ''],
    ['  K1 = Σ(отгрузки за месяц M) / Σ(отгрузки за месяц M-1)', ''],
    ['  Для проектов, у которых последний месяц реализации = M-1', ''],
    ['', ''],
    ['Коэффициент K2 (пролонгация во второй месяц):', ''],
    ['  K2 = Σ(отгрузки за месяц M) / Σ(отгрузки за месяц M-2)', ''],
    ['  Для проектов, у которых:', ''],
    ['    - последний месяц реализации = M-2', ''],
    ['    - НЕТ отгрузки в месяце M-1', ''],
    ['    - ЕСТЬ отгрузка в месяце M', ''],
    ['', ''],
    ['ИСТОЧНИКИ ДАННЫХ', ''],
    ['', ''],
    ['  1. prolongations.csv — приоритетный источник:', ''],
    ['     • last_month — последний месяц проекта', ''],
    ['     • AM — аккаунт-менеджер', ''],
    ['', ''],
    ['  2. financial_data.csv — финансовые данные:', ''],
    ['     • Суммы отгрузок по месяцам', ''],
    ['     • Account — менеджер (вторичный источник)', ''],
    ['', ''],
    ['ОСОБЕННОСТИ ОБРАБОТКИ', ''],
    ['', ''],
    ['  • Проекты со "стоп"/"end" исключаются из расчёта', ''],
    ['  • Дубли по id агрегируются (суммируются отгрузки)', ''],
    ['  • "в ноль" трактуется как 0 отгрузка', ''],
    ['  • Годовые коэффициенты — средневзвешенные по знаменателю', ''],
    ['    (Σnumerator / Σdenominator)', ''],
    ['', ''],
    ['ЦВЕТОВАЯ ИНДИКАЦИЯ', ''],
    ['', ''],
    [' Зелёный — коэффициент выше среднего по отделу', ''],
    [' Красный — коэффициент ниже среднего по отделу', ''],
]

for row_idx, row_data in enumerate(methodology_text, start=2):
    for col_idx, value in enumerate(row_data, start=1):
        if value:
            ws_method.cell(row=row_idx, column=col_idx, value=value)

ws_summary.auto_filter.ref = ws_summary.dimensions
ws_managers.auto_filter.ref = ws_managers.dimensions
ws_dynamics.auto_filter.ref = ws_dynamics.dimensions

# сохранение
wb.save('Отчет_Пролонгации_2023.xlsx')